# Guide pour l'Appel à une API Hugging Face pour la Segmentation d'Images

Bienvenue ! Ce notebook a pour but de vous guider pas à pas dans l'utilisation de l'API d'inférence de Hugging Face pour effectuer de la segmentation d'images. La segmentation d'images consiste à attribuer une étiquette (comme "cheveux", "vêtement", "arrière-plan") à chaque pixel d'une image.

Nous allons :
1. Comprendre ce qu'est une API et comment s'y connecter.
2. Envoyer une image à un modèle de segmentation hébergé sur Hugging Face.
3. Récupérer et interpréter les résultats.
4. Visualiser les masques de segmentation.
5. Étendre cela pour traiter plusieurs images.

## 1. Configuration Initiale et Importations

Commençons par importer les bibliothèques Python nécessaires. Nous aurons besoin de :
- `os` pour interagir avec le système de fichiers (lister les images).
- `requests` pour effectuer des requêtes HTTP vers l'API.
- `PIL (Pillow)` pour manipuler les images.
- `matplotlib.pyplot` pour afficher les images et les masques.
- `numpy` pour la manipulation des tableaux (les images sont des tableaux de pixels).
- `tqdm.notebook` pour afficher une barre de progression (utile pour plusieurs images).
- `base64` et `io` pour décoder les masques renvoyés par l'API.

In [ ]:
import os, mimetypes
from pathlib import Path

import requests
from PIL import Image
import re

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

import numpy as np
from tqdm import tqdm
import base64
import time


from dotenv import load_dotenv
import os
import io


### Variables de Configuration

Nous devons définir quelques variables :
- `image_dir`: Le chemin vers le dossier contenant vos images. **Assurez-vous de modifier ce chemin si nécessaire.**
- `max_images`: Le nombre maximum d'images à traiter (pour ne pas surcharger l'API ou attendre trop longtemps).
- `api_token`: Votre jeton d'API Hugging Face. **IMPORTANT : Gardez ce jeton secret !**

**Comment obtenir un token API Hugging Face ?**
1. Créez un compte sur [huggingface.co](https://huggingface.co/).
2. Allez dans votre profil -> Settings -> Access Tokens.
3. Créez un nouveau token (par exemple, avec le rôle "read").
4. Copiez ce token ici.

In [ ]:
# TODO: Modifiez ces valeurs selon votre configuration
image_dir = r"..\data\IMG"  # Exemple : si vous êtes sur Colab et avez uploadé un dossier
max_images = 5 # Commençons avec peu d'images
Masque_ref_dir = r"..\data\Mask"

# Charge automatiquement le fichier .env (à la racine ou dans le dossier courant)
load_dotenv()

# Récupère le token
token = os.getenv("TOKEN_HUG")

if token == "":
    print("\nATTENTION : Vous devez remplacer 'VOTRE_TOKEN' par votre token API personnel.")
else:
    print ("Token_ok")


### ✅ Contrôle de l'existence des fichiers et répertoires

In [ ]:
if not os.path.exists(image_dir):
    os.makedirs(image_dir)
    print(f"Dossier '{image_dir}' créé. Veuillez y ajouter des images .jpg ou .png.")
else:
    print(f"Dossier '{image_dir}' existant.")

if not os.path.exists(Masque_ref_dir):
    os.makedirs(Masque_ref_dir)
    print(f"Dossier '{Masque_ref_dir}' créé. Veuillez y ajouter les masques de référence.")
else:
    print(f"Dossier '{Masque_ref_dir}' existant.")

## 2. Comprendre l'API d'Inférence Hugging Face

L'API d'inférence permet d'utiliser des modèles hébergés sur Hugging Face sans avoir à les télécharger ou à gérer l'infrastructure.

- **Modèle utilisé** : Nous allons utiliser le modèle `sayeed99/segformer_b3_clothes`, spécialisé dans la segmentation de vêtements et de parties du corps.
- **URL de l'API** : L'URL pour un modèle est généralement `https://api-inference.huggingface.co/models/NOM_DU_MODELE`.
- **Headers (En-têtes)** : Pour s'authentifier et spécifier le type de contenu, nous envoyons des en-têtes avec notre requête.
    - `Authorization`: Contient notre token API (précédé de `Bearer `).
    - `Content-Type`: Indique que nous envoyons une image au format JPEG (ou PNG selon le cas).

In [ ]:
#API_URL = "https://api-inference.huggingface.co/models/yeed99/segformer_b3_clothes" # Remplacez ... par le bon endpoint.
API_URL = "https://router.huggingface.co/hf-inference/models/sayeed99/segformer_b3_clothes"
headers = {
    "Authorization": f"Bearer {token}"
    # Le "Content-Type" sera ajouté dynamiquement lors de l'envoi de l'image
}

### ✅ Préparation des fichiers d’images et de masques


**Cette étape permet de récupérer automatiquement les images d’entrée ainsi que leurs masques de référence, puis de vérifier qu'ils soient bien présentes dans les dossiers correspondants.**

**Ce que fait le code :**
- Parcourir les dossiers contenant les images et les masques
- Sélectionner uniquement les fichiers image (.png, .jpg, .jpeg, .gif)
- Trier les fichiers avec un "tri naturel" (exemple : image_2 est placé avant image_10)
- Limiter le nombre de fichiers chargés à max_images
- Affiche un message si aucun fichier n’est trouvé

**Le code construit deux listes synchronisées :**
- <i>image_paths</i> → chemins des images à traiter
- <i>Masque_ref_paths</i> → chemins des masques associés

<p>Ces chemins serviront à charger correctement les images pour la suite du traitement.</p>

In [ ]:
def tri_naturel(path):
    nom_fichier = os.path.basename(path)
    return [int(s) if s.isdigit() else s for s in re.split(r'(\d+)', nom_fichier)]

#toutes les images
toutes_les_images = [
    os.path.join(image_dir, f)
    for f in os.listdir(image_dir)
    if os.path.isfile(os.path.join(image_dir, f))
    and f.lower().endswith((".png", ".jpg", ".jpeg", ".gif"))
]

if not toutes_les_images : print(f"Aucune image trouvée dans '{image_dir}'. Veuillez y ajouter des images.")

#tous les masques 
tous_les_masques = [
    os.path.join(Masque_ref_dir, f) # pour avoir un chemin sans erreur
    for f in os.listdir(Masque_ref_dir) # pour chaque elements contenu dans mon repertoire
    if os.path.isfile(os.path.join(Masque_ref_dir, f)) 
    and f.lower().endswith((".png", ".jpg", ".jpeg", ".gif")) # si c'est un fichier et que c'est une image
]

if not tous_les_masques : print(f"Aucun masque de reference trouvé dans '{Masque_ref_dir}'. Veuillez en ajouter.")

#tri naturel
image_paths  = sorted(toutes_les_images, key=tri_naturel)[:max_images]
Masque_ref_paths = sorted(tous_les_masques, key=tri_naturel)[:max_images]



## 3. Fonctions Utilitaires pour le Traitement des Masques

Le modèle que nous utilisons (`sayeed99/segformer_b3_clothes`) renvoie des masques pour différentes classes (cheveux, chapeau, etc.). Ces masques sont encodés en base64. Les fonctions ci-dessous sont fournies pour vous aider à :
1.  `CLASS_MAPPING`: Un dictionnaire qui associe les noms de classes (ex: "Hat") à des identifiants numériques.
2.  `get_image_dimensions`: Récupérer les dimensions d'une image.
3.  `decode_base64_mask`: Décoder un masque de base64 en une image (tableau NumPy) et le redimensionner.
4.  `create_masks`: Combiner les masques de toutes les classes détectées en un seul masque de segmentation final, où chaque pixel a la valeur de l'ID de sa classe.

**Cette partie est donnée car elle est spécifique au format de sortie de ce modèle et un peu complexe pour une première approche.** Lisez-la pour comprendre son rôle, mais ne vous attardez pas sur les détails d'implémentation pour l'instant.

In [ ]:
CLASS_MAPPING = {
    "Background": 0,
    "Hat": 1,
    "Hair": 2,
    "Sunglasses": 3,
    "Upper-clothes": 4,
    "Skirt": 5,
    "Pants": 6,
    "Dress": 7,
    "Belt": 8,
    "Left-shoe": 9,
    "Right-shoe": 10,
    "Face": 11,
    "Left-leg": 12,
    "Right-leg": 13,
    "Left-arm": 14,
    "Right-arm": 15,
    "Bag": 16,
    "Scarf": 17
}

CLASS_NAMES = {
    1: "Hat",
    2: "Hair",
    3: "Sunglasses",
    4: "Upper-clothes",
    5: "Skirt",
    6: "Pants",
    7: "Dress",
    8: "Belt",
    9: "Left-shoe",
    10: "Right-shoe",
    11: "Face",
    12: "Left-leg",
    13: "Right-leg",
    14: "Left-arm",
    15: "Right-arm",
    16: "Bag",
    17: "Scarf"
}

CLASS_COLORS = {
    0: (0, 0, 0, 0),         # Background transparent
    1: (0, 255, 255, 120),   # Jaune - Hat
    2: (0, 165, 255, 120),   # Orange - Hair
    3: (255, 0, 255, 120),   # Magenta - Sunglasses
    4: (0, 0, 255, 120),     # Rouge - Upper-clothes
    5: (255, 255, 0, 120),   # Cyan - Skirt
    6: (0, 255, 0, 120),     # Vert - Pants
    7: (255, 0, 0, 120),     # Bleu - Dress
    8: (128, 0, 128, 120),   # Violet - Belt
    9: (0, 255, 255, 120),   # Jaune - Left-shoe
    10: (255, 140, 0, 120),  # Orange foncé - Right-shoe
    11: (222, 184, 135, 120),   # Face - beige clair
    12: (210, 180, 140, 120),   # Left-leg - beige moyen
    13: (205, 170, 125, 120),   # Right-leg - beige chaud
    14: (218, 165, 105, 120),   # Left-arm - beige doré
    15: (244, 164, 96, 120),    # Right-arm - beige cuivré
    16: (0, 128, 255, 120),  # Bleu clair - Bag
    17: (255, 20, 147, 120)  # Rose - Scarf
}

# Création groupe de classes pour futur classement  
SMALL_CLASSES = {1, 3, 8, 9, 10, 17}        # hat, sunglasses, belt, left/right-shoe, scarf
LIMB_CLASSES  = {12,13,14,15}               # legs/arms
TORSO_HEAD    = {4,6,7,11}                  # upper, pants, dress, face

def get_image_dimensions(img_path):
    original_image = Image.open(img_path)
    return original_image.size

def decode_base64_mask(base64_string, width, height):
    mask_data = base64.b64decode(base64_string)
    mask_image = Image.open(io.BytesIO(mask_data))
    mask_array = np.array(mask_image)

    if len(mask_array.shape) == 3:
        mask_array = mask_array[:, :, 0]  # Take first channel if RGB
    mask_image = Image.fromarray(mask_array).resize((width, height), Image.NEAREST)
    return np.array(mask_image)

def create_masks(results, width, height):
    combined_mask = np.zeros((height, width), dtype=np.uint8)  # Initialize with Background (0)
    # Process non-Background masks first

    for result in results:
        label = result['label']
        class_id = CLASS_MAPPING.get(label, 0)
        if class_id == 0:  # Skip Background
            continue
        mask_array = decode_base64_mask(result['mask'], width, height)
        combined_mask[mask_array > 0] = class_id

    # Process Background last to ensure it doesn't overwrite other classes unnecessarily
    # (Though the model usually provides non-overlapping masks for distinct classes other than background)
    for result in results:
        if result['label'] == 'Background':
            mask_array = decode_base64_mask(result['mask'], width, height)
            # Apply background only where no other class has been assigned yet
            # This logic might need adjustment based on how the model defines 'Background'
            # For this model, it seems safer to just let non-background overwrite it first.
            # A simple application like this should be fine: if Background mask says pixel is BG, set it to 0.
            # However, a more robust way might be to only set to background if combined_mask is still 0 (initial value)
            combined_mask[mask_array > 0] = 0 # Class ID for Background is 0
    
    return combined_mask

### ✅ Fonctions de mise en couleur d'un masque image
<span>à partir de la classe CLASS_NAMES</span> 

In [ ]:
def Colore_Mask (Masque_initial, hauteur, largeur) :
    # masque RGBA vide (transparents)
    Masque_retour = np.zeros((hauteur, largeur, 4), dtype=np.uint8)

    # ne colorer que les classes définies dans CLASS_COLORS
    for class_id, color in CLASS_COLORS.items():
        Masque_retour[Masque_initial == class_id] = color

    return Masque_retour

### Alignement du masque de référence avec le masque prédictif

Cette fonction garantit que le masque de référence a la même résolution que le masque prédit par le modèle.

**Pourquoi ?**
Pour comparer les deux masques pixel par pixel (IoU, Dice, Accuracy), ils doivent être parfaitement alignés.

**Que fait le code :**
- comparer les dimensions du masque référence et du masque prédiction
- Si elles diffèrent, redimensionner le masque référence
  en utilisant `NEAREST` → préserve les classes (pas de flou ni interpolation erronée)
- Sinon, retourner le masque original

In [ ]:
def alignement_masque_prediction(masque_reference: np.ndarray, masque_prediction: np.ndarray):

    assert masque_reference.ndim == 2 or masque_reference.ndim == 3, "Masque référence invalide"
    assert masque_prediction.ndim == 2 or masque_prediction.ndim == 3, "Masque prédiction invalide"

    if masque_reference.shape[:2] != masque_prediction.shape[:2]:
        return np.array(
            Image.fromarray(gt).resize((masque_prediction.shape[1], masque_prediction.shape[0]), Image.NEAREST)
        )

    return masque_reference

### Mesures IoU & Dice (multiclasse)

Nous calculons IoU et Dice par classe à partir des 2 masques (Reference et Predit) :

- **IoU**

$$
\text{IoU} = \frac{Intersection}{Union}
$$

- **Dice** 
$$
\text{Dice} = \frac{2·\text{Intersection}}{\text{Taille du masque de Référence} + \text{Taille du masque Prédit}}
$$

Options :
- `average="all"` : moyenne sur toutes les classes (hors fond 0)
- `average="present"` : moyenne uniquement sur les classes présentes dans le masque de référence et le masque prédit (Union != 0)

La classe **0 (fond)** est ignorée dans les moyennes pour refléter l’évaluation des vêtements.


In [ ]:
def iou_dice_multiclass(
    Masque_pred: np.ndarray,
    Masque_ref: np.ndarray,
    num_classes_vetement: int = len(CLASS_COLORS),  # nb total de classes (incluant 0)
    average: str = "all",                           # "all" ou "present"
    eps: float = 1e-9
):
    assert Masque_pred.shape == Masque_ref.shape, "les 2 masques doivent etre de la même taille."
    assert average in {"all", "present"}, "average doit être 'all' ou 'present'"
    
    # Classes effectivement PRÉDITES (hors fond 0)
    classes_predites = np.setdiff1d(np.unique(Masque_pred), [0])
    classes_predites_ref = np.setdiff1d(np.unique(Masque_ref), [0])
    
    classes_communes = np.intersect1d(classes_predites, classes_predites_ref)
    
    # Tableaux résultats indexés par id de classe (taille = nb total de classes)
    ious  = np.zeros(num_classes_vetement, dtype=np.float64)
    dices = np.zeros(num_classes_vetement, dtype=np.float64)
    
    # Calcul par classe prédite
    for Vetement in classes_communes:
        pred_vetement = (Masque_pred == Vetement)
        ref_vetement  = (Masque_ref  == Vetement)

        intersection = np.logical_and(pred_vetement, ref_vetement).sum()
        union        = np.logical_or (pred_vetement, ref_vetement).sum()

        total        = pred_vetement.sum() + ref_vetement.sum()

        ious[Vetement]  = intersection / (union + eps)
        dices[Vetement] = (2.0 * intersection) / (total + eps)
    
    if intersection != 0 :
        # Tables uniquement pour les classes PRÉDITES
        if classes_communes.size > 0:
            table_Ious  = np.column_stack((classes_communes, ious[classes_communes]))
            table_Dices = np.column_stack((classes_communes, dices[classes_communes]))
        else:
            table_Ious = table_Dices = np.empty((0, 2), dtype=np.float64)

        # Moyennes
        if average == "present":
            # moyenne sur les classes PRÉDITES (si aucune, 0.0)
            mean_iou  = float(ious [classes_communes].mean())  if classes_communes.size else 0.0
            mean_dice = float(dices[classes_communes].mean())  if classes_communes.size else 0.0
        else:
            # moyenne sur TOUTES les classes (hors fond 0), y compris non prédites
            mean_iou  = float(ious [1:].mean()) if num_classes_vetement > 1 else 0.0
            mean_dice = float(dices[1:].mean()) if num_classes_vetement > 1 else 0.0

    return table_Ious, mean_iou, table_Dices, mean_dice

### -- fonctions d'exploitation des résultats ---

In [ ]:
def moyenne_total(Tableau):
    """Moyenne brute sur toutes les classes (classe absente = 0 inclus)"""
    return float(np.mean(Tableau)) if Tableau.size else 0.0

def moyenne_presente(Tableau, support):
    """Moyenne uniquement sur les classes présentes (support > 0)"""
    m = (support > 0)
    return float(np.mean(Tableau[m])) if m.any() else 0.0

def moyenne_ponderee(Tableau, support):
    """Moyenne pondérée par le support (nb pixels dans le masque GT)"""
    m = (support > 0)
    if not m.any():
        return 0.0
    
    w = support[m].astype(float)
    w /= (w.sum() + 1e-9)    # normalisation poids présence
    return float((Tableau[m] * w).sum())

## 4. Segmentation d'une Seule Image

Avant de traiter toutes les images, concentrons-nous sur une seule pour bien comprendre le processus.

Étapes :
1.  Choisir une image.
2.  Ouvrir l'image en mode binaire (`"rb"`) et lire son contenu (`data`).
3.  Déterminer le `Content-Type` (par exemple, `"image/jpeg"` ou `"image/png"`).
4.  Envoyer la requête POST à l'API avec `requests.post()` en passant l'URL, les headers et les données.
5.  Vérifier le code de statut de la réponse. Une erreur sera levée si le code n'est pas 2xx (succès) grâce à `response.raise_for_status()`.
6.  Convertir la réponse JSON en un dictionnaire Python avec `response.json()`.
7.  Utiliser nos fonctions `get_image_dimensions` et `create_masks` pour obtenir le masque final.
8.  Afficher l'image originale et le masque segmenté.

### Pipeline de traitement d'une image (API Hugging Face + métriques segmentation)

#### **Cette section réalise le traitement complet d'une image pour évaluer un modèle de segmentation :**

* Chargement de l'image et du masque de référence :
    Lecture de l'image source, format pour l'appel API, chargement du masque de référence()
* Appel à l'API Hugging Face :
    Envoi de l'image binaire à l’API de segmentation et récupération de la prédiction du modèle
* Reconstruction du masque prédit : 
    Extraction du masque de segmentation et ajustement des dimensions si nécessaire pour correspondre au masque de référence
* Calcul des 2 métriques par classe et de leurs moyennes par image:
    **IoU** et **Dice score** sur les **classes prédites** présentes uniquement (hors Background)
* Colorisation du masque prédit et du masque de référence : 
     Conversion du masque de classes → masque RGBA, application des couleurs pour les classes **connues**, pixels des classes inconnues laissés transparents
* Export / Visualisation : Conversion du masque coloré en image affichable

In [ ]:
if image_paths:
    single_image_path = image_paths[0]
    print(f"Traitement de l'image : {single_image_path}")

    try:
        # --- Content-Type propre (via mimetypes) ---
        ext = Path(single_image_path).suffix.lower()
        content_type = mimetypes.types_map.get(ext, "application/octet-stream")
        print("Format détecté :", ext, "| Content-Type :", content_type)

        # --- Headers fusionnés proprement ---
        req_headers = {**headers, "Content-Type": content_type}

        # --- Lecture binaire image ---
        with open(single_image_path, "rb") as f:
            image_data = f.read()

        # --- Masque de référence (sans altérer les labels) ---
        Masque_ref_image = Image.open(Masque_ref_paths[0])
        Masque_ref_final = np.array(Masque_ref_image)

        # --- Appel API (sécurisé) ---
        Reponse_API = requests.post(API_URL, data=image_data, headers=req_headers, timeout=60)
        Reponse_API.raise_for_status()
        Reponse = Reponse_API.json()

        # --- Dimensions (ta fonction) ---
        width_Image, height_Image = get_image_dimensions(single_image_path)

        # --- Masque prédiction + alignement ref ---
        Masque_prediction_final = create_masks(Reponse, width_Image, height_Image)
        Masque_ref_final = alignement_masque_prediction(Masque_ref_final, Masque_prediction_final)

        # --- IoU / Dice (sur classes présentes prédictes) ---
        ious_par_vetement, Moyenne_iou, dices_par_vetement, Moyenne_dice = iou_dice_multiclass(
            Masque_prediction_final, Masque_ref_final, average="present"
        )

        # --- Colorisation (classes connues uniquement => le reste reste transparent) ---
        Masque_couleur = Colore_Mask(Masque_prediction_final, height_Image, width_Image)
        Masque_Image = Image.fromarray(Masque_couleur)
        #Masque_Image.show()

        Masque_couleur_ref = Colore_Mask(Masque_ref_final, height_Image, width_Image)
        Masque_ref = Image.fromarray(Masque_couleur_ref)
        #Masque_ref.show()

    except requests.HTTPError as e:
        print(f"Erreur HTTP API ({Reponse_API.status_code}) : {e}")
        print("Corps de réponse :", getattr(Reponse_API, "text", "")[:500])
    except Exception as e:
        print(f"Une erreur est survenue : {e}")

else:
    print("Aucune image à traiter. Vérifiez la configuration de 'image_dir' et 'max_images'.")

## 5. Segmentation de Plusieurs Images (Batch)

Maintenant que nous savons comment traiter une image, nous pouvons créer une fonction pour en traiter plusieurs.
Cette fonction va boucler sur la liste `image_paths` et appliquer la logique de segmentation à chaque image.
Nous utiliserons `tqdm` pour avoir une barre de progression.

### Processus de la fonction
<ol>
<li>Lire l'image + masque GT</li>
<li>Envoyer image à l'API</li>
<li>Reconstruire masque prédit</li>
<li>Aligner masque GT aux dimensions</li>
<li>Extraire classes présentes (sans 0)</li>
<li>Intersection classes prédiction ∩ GT</li>
<li>Calcul IoU / Dice par classe</li>
<li>Stocker résultats dans un DataFrame</li>
<li>Optionnel : coloriser masques en RGBA</li>
</ol>

In [ ]:
def segment_images_batch(list_of_image_paths, Masque_ref_paths, mode_segmentation="RGBA"):
    batch_segmentations = []
    batch_segmentations_ref = []
    Tableau_de_resultats = []

    try:
        total = min(len(list_of_image_paths), len(Masque_ref_paths))
        
        # Bar de défilement
        for image_path, mask_path in tqdm(
            zip(list_of_image_paths, Masque_ref_paths),
            desc="Traitement des images en cours",
            total=total,
        ):
            try:
                image_nom = os.path.basename(image_path)

                # --- récupération du Content-Type ---
                ext = Path(image_path).suffix.lower()
                content_type = mimetypes.types_map.get(ext, "application/octet-stream")

                # --- Lecture des 2 fichiers ---
                with open(image_path, "rb") as f:
                    image_data = f.read()
                masque_ref_final = np.array(Image.open(mask_path).convert("L"), dtype=np.int32)

                # --- Appel API ---
                Reponse_API = requests.post(
                    API_URL,
                    data=image_data,
                    headers={**headers, "Content-Type": content_type},
                    timeout=60,
                )
                
                # --- Gestion d'erreur ---
                if not (200 <= Reponse_API.status_code < 300):
                    print(f"Erreur {Reponse_API.status_code} : Image non traitée: {image_path}")
                    batch_segmentations.append(None)
                    continue

                Reponse = Reponse_API.json()
                time.sleep(0.5)  # throttle léger

                # --- Reconstruction des masques avec la même dimension---
                width_Image, height_Image = get_image_dimensions(image_path)
                masque_pred_final = create_masks(Reponse, width_Image, height_Image)
                masque_ref_final = alignement_masque_prediction(masque_ref_final, masque_pred_final)

                # --- Métriques (classes prédites et presentes dans le masque de référence uniquement) ---
                ious_par_vetement, Moyenne_iou, dices_par_vetement, Moyenne_dice = iou_dice_multiclass(
                    masque_pred_final, masque_ref_final, average="present"
                )

                # Mapping class_id -> métrique (plus simple que filtrer par masque à chaque fois)
                d_iou  = {int(cid): float(val) for cid, val in ious_par_vetement}
                d_dice = {int(cid): float(val) for cid, val in dices_par_vetement}
                
                classes_presentes_Masque = np.setdiff1d(np.unique(masque_pred_final), [0])
                classes_communes = np.intersect1d(ious_par_vetement, classes_presentes_Masque)

                # --- création du tableau de retour ---
                for c in classes_communes:
                    c = int(c)
                    support = int((masque_pred_final == c).sum())

                    Tableau_de_resultats.append([
                        image_nom,                    # image_id
                        c,                            # class_id
                        support,                      # support (pixels prédits)
                        d_iou.get(c, np.nan),         # IoU
                        d_dice.get(c, np.nan),        # Dice
                    ])
                   
                # --- Colorisation optionnelle pour affichage plus agréable---
                if mode_segmentation.upper() == "RGBA":
                    masque_affiche = Colore_Mask(masque_pred_final, height_Image, width_Image)
                    masque_ref_affiche = Colore_Mask(masque_ref_final, height_Image, width_Image)
                else:
                    masque_affiche = masque_pred_final  # gris (labels)
                    masque_ref_affiche = masque_ref_final

                # --- ajout dans batch retour pour affichage
                batch_segmentations.append(Image.fromarray(masque_affiche))
                batch_segmentations_ref.append(Image.fromarray(masque_ref_affiche))

            except Exception as e:
                print(f"Erreur : Image non traitée: {image_path} → {e}")
                batch_segmentations.append(None)
                batch_segmentations_ref.append(None)
                continue

        # --- DataFrame résultats ---
        dataframe_resultats = pd.DataFrame(
            Tableau_de_resultats,
            columns=["Nom_Image", "Classe_vetement", "Support", "IoU", "Dice"]
            )
        
        return batch_segmentations, batch_segmentations_ref, dataframe_resultats

    except Exception as e:
        print(f"Erreur : Fichier  → {e}")
        return batch_segmentations, batch_segmentations_ref, pd.DataFrame(columns=["Nom_Image", "Classe_vetement", "Support", "IoU", "Dice"])



# Appeler la fonction pour segmenter les images listées dans image_paths
if image_paths:
    print(f"\nTraitement de {len(image_paths)} image(s) en batch...")
    batch_seg_results, batch_seg_ref, Resultats = segment_images_batch(image_paths, Masque_ref_paths, mode_segmentation="RGBA") ## ici gray ou RGBA (par defaut)
    print("Traitement en batch terminé.")
else:
    batch_seg_results = []
    batch_seg_ref = []
    Resultats = []
    
    print("Aucune image à traiter en batch.")


## 6. Affichage des Résultats en Batch

Nous allons maintenant créer une fonction pour afficher les images originales et leurs segmentations correspondantes côte à côte, dans une grille.

## display_segmented_images_batch — Qu’est-ce qu’elle fait ?

<p>
<strong>But :</strong> afficher, pour chaque image, un panneau de 4 vues alignées :
<em>(1)</em> l’image originale, <em>(2)</em> le masque prédit, <em>(3)</em> le masque de référence, <em>(4)</em> la fusion image+masque.
La fonction ajoute aussi des <strong>légendes analytiques</strong> (IoU/Dice par classe + moyennes).
</p>
<p>
<span style="white-space:nowrap">Entrées (arguments)</span>
<ol>
<li><code>original_image_paths</code> : liste des chemins des images sources (ordre = vérité unique du batch).</li>
<li><code>segmentation_masks</code> : liste des masques prédits (images PIL ou arrays), <em>None</em> si échec.</li>
<li><code>segmentation_mask_ref</code> : liste des masques de référence (PIL/arrays), <em>None</em> si absent.</li>
<li><code>dataframe_resultats</code> : DataFrame avec colonnes
<code>["Nom_Image","Classe_vetement","Support","IoU","Dice"]</code> (une ligne par classe et image).</li>
<li><code>masque_segmentation</code> : <em>"RGBA"</em> pour affichage couleur (overlay alpha), sinon niveaux de gris + overlay rouge translucide en fusion.</li>
</ol>

### Pipeline (étapes principales)
<ol>
<li><strong>Initialisation figure</strong> : pour chaque image, création d’une figure à 4 colonnes.</li> 
<li><strong>Affichage original</strong> : ouverture du fichier et <code>imshow()</code> sans axes.</li> 
<li><strong>Affichage masque prédit</strong> : <ul>
    <li>Si <code>RGBA</code> → rendu direct couleur.</li>
    <li>Sinon → rendu en niveaux de gris (<code>cmap="gray"</code>).</li>
    <li>Si masque manquant → message “Aucun masque”.</li> </ul>
</li> 
<li><strong>Affichage masque de référence</strong> : même logique que le prédit.</li>
<li><strong>Fusion image + masque (predit)</strong> : <ul>
    <li><em>Mode RGBA</em> : <code>alpha_composite</code> après resize éventuel au nearest.</li>
    <li><em>Mode gris</em> : création d’un <strong>overlay rouge semi-transparent</strong> où le masque &gt; 0.</li></ul>
</li>
<li><strong>Panneaux de légendes</strong> : <ul>
    <li>Dans l’axe “masque prédit” : table texte monospace “<em>Résultats par classe</em>” triée par IoU décroissant.</li>
    <li>Dans l’axe “fusion” : <strong>récap</strong> des moyennes (mIoU et mDice) — total, présentes, pondérées — avec un encadré coloré (vert/gris/rouge) selon la qualité (mIoU présente).</li> </ul>
    </li>
<li><strong>Layout & rendu</strong> : <code>tight_layout()</code> puis <code>show()</code>.</li>
</ol>

### Légendes & métriques (détail)

<strong>Résultats par classe</strong> (sur l’axe du masque prédit) : pour chaque ligne du DataFrame de l’image courante où
<code>Support > 0</code>, on affiche
<code>Classe_vetement → IoU, Dice</code> (formatés en 3 décimales).
Le nom lisible provient de <code>CLASS_NAMES</code> (fallback “Classe {id}”).

<strong>Récap moyennes</strong> (sur l’axe de fusion) :
<code>mIoU</code> et <code>mDice</code> sont calculés via tes fonctions <code>moyenne_total</code>, <code>moyenne_presente</code>, <code>moyenne_ponderee</code>.
L’encadré passe en <span style="color:#228B22"><strong>vert</strong></span> si ≥ 0.70, <span>gris</span> si ≥ 0.40, sinon <span style="color:#B22222"><strong>rouge</strong></span>.

### Comportements robustes

<strong>Tailles différentes</strong> (fusion RGBA) : le masque est <em>resizé en nearest</em> à la taille de l’image.
<strong>Objets PIL ou NumPy</strong> : conversion sûre (e.g., <code>Image.fromarray(...)</code> si nécessaire).
<strong>Absence de masque</strong> : message explicite et pas de crash.
<strong>Lisibilité</strong> : axes nommés (<code>ax_img</code>, <code>ax_pred</code>, <code>ax_ref</code>, <code>ax_fuse</code>), titres courts, police monospace pour les panneaux.

### Résultat visuel (par image)
<table> <tr> <th>Original</th> <th>Masque prédit</th> <th>Masque référence</th> <th>Fusion + métriques</th> </tr> <tr> <td align="center">photo</td> <td align="center">labels/couleurs</td> <td align="center">labels/couleurs</td> <td align="left">overlay + moyennes + cadre couleur</td> </tr> </table>

In [ ]:
def display_segmented_images_batch(original_image_paths, segmentation_masks, segmnetation_mask_ref, dataframe_resultats, masque_segmentation="RGBA"):
    nombre_lignes = len(original_image_paths)
    nombre_colonnes = 4
    nombre_images_par_graph = 1

    ### BOUCLE GENERALE
    for ligne_en_cours in range(0,nombre_lignes,nombre_images_par_graph) :
        
        ### Affiche les images originales et leurs masques segmentés.
        nombre_lignes_affichee = nombre_images_par_graph
        
        ### je definis l'organisation de mes images
        plt.figure(figsize=(nombre_colonnes * 10, nombre_lignes_affichee * 10))
        plt.subplots_adjust(top=0.95, bottom=0.05, hspace=0.35, wspace=0.05)

        ###DOUBLE BOUCLE POUR AFFICHAGE
        for ligne_affichee in range(nombre_lignes_affichee) :

            # IMAGE DEPART
            plt.subplot(nombre_lignes_affichee, nombre_colonnes, ligne_affichee * 3 + 1)  # position du sous-graphe
            image_affichee = Image.open(original_image_paths[ligne_en_cours])
            plt.imshow(image_affichee)

            plt.title(f"Image {ligne_en_cours}")
            plt.axis("off") #retire le quadrillage

            # MASQUE COULEUR ou NUANCE DE GRIS
            ax3 = plt.subplot(nombre_lignes_affichee, nombre_colonnes, ligne_affichee * 3 + 2)
            mask_affiche = segmentation_masks[ligne_en_cours]
            
            if mask_affiche is not None:
                # Affiche le mask soit couleur
                if masque_segmentation.upper() == "RGBA" :
                    plt.imshow(mask_affiche)
                else :
                    #soit rouge sur image 
                    plt.imshow(mask_affiche,cmap="gray")
                    
                plt.title(f"Masque segmenté {ligne_en_cours}",color="green")
            else:
                plt.text(0.5, 0.5, "Aucun masque", ha="center", va="center", color="red")
                plt.title("Erreur / Manquant", color="red")
            
            plt.axis("off")

            # MASQUE COULEUR ou NUANCE DE GRIS
            ax3 = plt.subplot(nombre_lignes_affichee, nombre_colonnes, ligne_affichee * 3 + 3)
            mask_ref_affiche = segmnetation_mask_ref[ligne_en_cours]
            
            if mask_ref_affiche is not None:
                # Affiche le mask soit couleur
                if masque_segmentation.upper() == "RGBA" :
                    plt.imshow(mask_ref_affiche)
                else :
                    #soit rouge sur image 
                    plt.imshow(mask_ref_affiche,cmap="gray")
                    
                plt.title(f"Masque de référence segmenté {ligne_en_cours}",color="green")
            else:
                plt.text(0.5, 0.5, "Aucun masque", ha="center", va="center", color="red")
                plt.title("Erreur / Manquant", color="red")
            
            plt.axis("off")
        
            #### FUSION IMAGE + MASQUE (SI NUANCES DE GRIS MASQUE = NUANCE DE ROUGE SUR LA FUSION)
            if mask_affiche is not None:
                ax4 = plt.subplot(nombre_lignes_affichee, nombre_colonnes, ligne_affichee * 3 + 4)
                
                if masque_segmentation.upper() == "RGBA" :
                    # convertion dans le meme format pour la fusion
                    image_RGBA = image_affichee.convert("RGBA")
                    mask_RGBA = mask_affiche.convert("RGBA")

                    # Verification des tailles des images et des mask
                    if mask_RGBA.size != image_RGBA.size:
                        mask_RGBA= mask_RGBA.resize(image_RGBA.size, resample=Image.NEAREST)

                    # la taille est identique donc fusion
                    fusion_images = Image.alpha_composite(image_RGBA, mask_RGBA)
                else :
                    image_RGBA = image_affichee.convert("RGBA")
                    mask_L =  mask_affiche.convert("L").resize(image_RGBA.size, Image.NEAREST)
                    
                    #Échelle le masque: 0 -> 0 (transparent), >0 -> opacité fixe (ex: 120/255)
                    OPACITE = 120  # ajuste entre 0 et 255
                    mask_alpha = mask_L.point(lambda p: OPACITE if p > 0 else 0)

                    #Je Crée un masque en rouge transparent
                    overlay = Image.new("RGBA", image_RGBA.size, (255, 0, 0, 0))  # rouge transparent
                    #j'y applique mon mask
                    overlay.putalpha(mask_alpha)  # masque devient alpha

                    # Fusion finale
                    fusion_images = Image.alpha_composite(image_RGBA, overlay)

                # Affichage fusion
                plt.imshow(fusion_images)
                plt.title(f"Fusion des images {ligne_en_cours}",color="red")

            else :
                plt.text(0.5, 0.5, "Pas de fusion", ha="center", va="center", color="red")
                plt.title("Erreur / Manquant", color="red")

            plt.axis("off")
            ## FIN FUSION

            ## CREATION LEGENDES
            nom_image= os.path.basename(original_image_paths[ligne_en_cours])
            dataframe_img = dataframe_resultats[dataframe_resultats["Nom_Image"] == nom_image]
                        
            if not dataframe_img.empty:
                
                ## LEGENDE RESULTATS PAR CLASSE, SUPPORT PRESENT SUR L'IMAGE DE DEPART
                df_sorted = dataframe_img.sort_values("IoU", ascending=False)
                lignes = ["Résultats par classe :"]

                for _, row in df_sorted.iterrows():
                    if row["Support"] > 0:
                        class_id = int(row["Classe_vetement"])
                        class_name = CLASS_NAMES.get(class_id, f"Class {class_id}")
                        lignes.append(f"{class_name:<15}  IoU={row['IoU']:.3f}  Dice={row['Dice']:.3f}")
                        txt_classes = "\n".join(lignes)

                ax3.text(
                    0.10, 0.98, txt_classes,
                    transform=ax3.transAxes,
                    va="top", ha="right",
                    fontsize=10,
                    family="monospace",
                    bbox=dict(facecolor="white", alpha=0.85, edgecolor="gray")
                    )
                
                ## LEGENDE RESULTATS MOYENNES
                iou  = dataframe_img["IoU"].to_numpy(dtype=float)
                dice = dataframe_img["Dice"].to_numpy(dtype=float)
                sup  = dataframe_img["Support"].to_numpy(dtype=float)

                mIoU_tot = moyenne_total(iou)
                mIoU_pres = moyenne_presente(iou, sup)
                mIoU_pond = moyenne_ponderee(iou, sup)
                mDice_tot = moyenne_total(dice)
                mDice_pres = moyenne_presente(dice, sup)
                mDice_pond = moyenne_ponderee(dice, sup)

                txt = (
                    f"{nom_image}\n"
                    f"mIoU Total/Présent/Pondéré : {mIoU_tot:.3f} / {mIoU_pres:.3f} / {mIoU_pond:.3f}\n"
                    f"mDice Total/Présent/Pondéré: {mDice_tot:.3f} / {mDice_pres:.3f} / {mDice_pond:.3f}\n"
                    f"Classes présentes sur masque prédit: {(sup>0).sum()} / {len(sup)}"
                )
                
                ### MISE EN AVANT PHOTOS LES + PERTINENTES
                if mIoU_pres >= 0.7 :
                    Cadre_couleur ="green"
                    epaisseur_cadre = 5
                elif mIoU_pres >= 0.4 :
                    Cadre_couleur ="gray"
                    epaisseur_cadre = 1
                else :
                    Cadre_couleur ="red"
                    epaisseur_cadre = 5
                    
                ax4.text(
                    0.98,
                    0,  txt,
                    transform=ax4.transAxes,
                    va="top", ha="right",
                    fontsize=11,
                    bbox=dict(facecolor="white", alpha=0.9, linewidth= epaisseur_cadre, edgecolor=Cadre_couleur)
                    )     
            
            ## COMPTEUR MANUEL POUR AFFICHAGE DES IMAGES EN X LIGNES
            ligne_en_cours += 1
            if ligne_en_cours > len(original_image_paths)-1 :
                break

        ## AFFICHAGE X LIGNES DE 4 IMAGES AVEC LEGENDES
        plt.subplots_adjust()
        plt.show()

    # Args:
    #     original_image_paths (list): Liste des chemins des images originales.
    #     segmentation_masks (list): Liste des masques segmentés (NumPy arrays).
        
if batch_seg_results:
    display_segmented_images_batch(image_paths, batch_seg_results, batch_seg_ref, Resultats, masque_segmentation="RGBA") ## ici gray ou RGBA (par defaut)
else:
    print("Aucun résultat de segmentation à afficher.")
    

## Conclusion et Prochaines Étapes

Félicitations ! Vous avez appris à :
- Configurer les appels à l'API d'inférence Hugging Face.
- Envoyer des images pour la segmentation.
- Interpréter les résultats (avec l'aide des fonctions fournies).
- Visualiser les segmentations.

Pistes d'amélioration ou d'exploration :
- **Gestion d'erreurs plus fine** : Implémenter des tentatives multiples (retry) en cas d'échec de l'API (par exemple, si le modèle est en cours de chargement).
- **Appels asynchrones** : Pour un grand nombre d'images, des appels asynchrones (avec `asyncio` et `aiohttp`) seraient beaucoup plus rapides.
- **Autres modèles** : Explorer d'autres modèles de segmentation ou d'autres tâches sur Hugging Face Hub.

N'hésitez pas à modifier le code, à tester avec vos propres images et à explorer davantage !

**_Note_** : Si vous aimez ce modèle, n'hésitez pas à le [télécharger](https://huggingface.co/sayeed99/segformer_b3_clothes) et jouer avec directement sur votre machine !

## Performance globale du modèle (Classes présentes vs Pondérées)

Ce graphique compare la performance moyenne du modèle selon deux méthodes d’agrégation :

| Méthode | Signification |
|--------|---------------|
| **Présentes** | moyenne sur les classes réellement présentes dans chaque image |
| **Pondérée** | moyenne pondérée par la surface (pixels) de chaque classe |

Deux métriques sont affichées :

- **mIoU** (Intersection over Union)
- **mDice** (Score de Dice — similarité)

**Objectif du graphique :**
- Vérifier la capacité générale du modèle à segmenter les vêtements
- Comparer les performances brutes (présentes) avec celles pondérées par taille

**Lecture :**
- Valeurs proches de **1.0** → très bonne segmentation
- mIoU ≤ mDice (normal, Dice est plus indulgent)
- Écart "Présente vs Pondérée" → sensibilité aux petites classes

<p><i>Une forte mIoU/mDice pondérée signifie que le modèle performe surtout sur les **zones importantes** (ex : vêtements principaux).</i></p>


In [ ]:
# --- Préparation ---
df = Resultats.copy()
df["IoU"]     = df["IoU"].astype(float)
df["Dice"]    = df["Dice"].astype(float)
df["Support"] = df["Support"].astype(float)

# --- Moyennes globales ---
mIoU_all       = moyenne_total(df["IoU"])
mIoU_present   = moyenne_presente(df["IoU"], df["Support"])
mIoU_weighted  = moyenne_ponderee(df["IoU"], df["Support"])

mDice_all      = moyenne_total(df["Dice"])
mDice_present  = moyenne_presente(df["Dice"], df["Support"])
mDice_weighted = moyenne_ponderee(df["Dice"], df["Support"])

# --- Résumés DataFrame ---
resume_mIoU_global = pd.DataFrame([{
    "mIoU_Total": mIoU_all,
    "mIoU_Presente": mIoU_present,
    "mIoU_Ponderee": mIoU_weighted,
    "nb_images": df["Nom_Image"].nunique(),
    "classes_presentes_moy": df.groupby("Nom_Image")["Support"].apply(lambda s: (s > 0).sum()).mean()
}])

resume_Dice_global = pd.DataFrame([{
    "mDice_Total": mDice_all,
    "mDice_Presente": mDice_present,
    "mDice_Ponderee": mDice_weighted,  # <-- correction d'orthographe
    "nb_images": df["Nom_Image"].nunique(),
    "classes_presentes_moy": df.groupby("Nom_Image")["Support"].apply(lambda s: (s > 0).sum()).mean()
}])

# --- Visuel synthétique (Présentes vs Pondérées) ---
def annotate_bars(ax, xs, vals, dy=0.02, fmt="{:.3f}"):
    for x, v in zip(xs, vals):
        ax.text(x, v + dy, fmt.format(v), ha="center", va="bottom", fontsize=10)

categories  = ["Moyenne (Classes présentes)", "Moyenne (Pondérée)"]
iou_scores  = [mIoU_present,  mIoU_weighted]
dice_scores = [mDice_present, mDice_weighted]

x = np.arange(len(categories))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, iou_scores,  width, label="mIoU",  color="blue")
plt.bar(x + width/2, dice_scores, width, label="mDice", color="green")

plt.xticks(x, categories)
plt.ylabel("Score moyen")
plt.ylim(0, 1.0)
plt.title("Résultats globaux du modèle de segmentation", fontsize=13, weight="bold")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)

annotate_bars(plt.gca(), x - width/2, iou_scores)
annotate_bars(plt.gca(), x + width/2, dice_scores)

plt.tight_layout()
plt.show()


## Performance par type de vêtement

Ce graphique présente, pour chaque classe vestimentaire :

- **IoU (Présente)** — qualité moyenne quand la classe apparaît
- **IoU (Pondérée)** — qualité pondérée par la surface
- **Dice (Présente)**
- **Dice (Pondérée)**

**Objectif du graphique :**
- Identifier les classes **faciles** vs **difficiles**
- Détecter les objets mal segmentés (souvent petits ou rares)

**Lecture :**
- Barres plus hautes = meilleure segmentation
- Écart Présente/Pondérée ➜ sensibilité à la taille des objets
- Haut du classement : vêtements larges (robe, haut, pantalon…)
- Bas du classement : petits accessoires (chaussures, chapeau, écharpe…)


<p><i>Ce graphique met en lumière les catégories à prioriser pour améliorer le modèle (souvent les petits objets).</i></p>


In [ ]:
# --- Filtre classes (on exclut la classe 0 si elle correspond au background) ---
df_cls = df[df["Classe_vetement"] != 0]

rows = []
for classe, g in df_cls.groupby("Classe_vetement"):
    iou  = g["IoU"].to_numpy()
    dice = g["Dice"].to_numpy()
    sup  = g["Support"].to_numpy()

    iou_pres  = moyenne_presente(iou, sup)
    iou_pond  = moyenne_ponderee(iou, sup)
    dice_pres = moyenne_presente(dice, sup)
    dice_pond = moyenne_ponderee(dice, sup)

    rows.append({
        "Classe_vetement": classe,
        "Classe": CLASS_NAMES.get(classe, f"Classe {classe}"),
        "Support_total": sup.sum(),
        "IoU_Presente": iou_pres,
        "IoU_Ponderee": iou_pond,
        "Dice_Presente": dice_pres,
        "Dice_Ponderee": dice_pond,
    })

df_moy = pd.DataFrame(rows).sort_values("IoU_Ponderee", ascending=False)

# --- Graphique comparatif (4 barres par classe) ---
x = np.arange(len(df_moy))
bar = 0.2

plt.figure(figsize=(15, 6))
plt.bar(x - 1.5*bar, df_moy["IoU_Presente"],  width=bar, label="IoU (Présente)",  color="#4e79a7")
plt.bar(x - 0.5*bar, df_moy["IoU_Ponderee"],  width=bar, label="IoU (Pondérée)", color="#a0cbe8")
plt.bar(x + 0.5*bar, df_moy["Dice_Presente"], width=bar, label="Dice (Présente)", color="#f28e2b")
plt.bar(x + 1.5*bar, df_moy["Dice_Ponderee"], width=bar, label="Dice (Pondérée)", color="#ffbe7d")

plt.xticks(x, df_moy["Classe"], rotation=45, ha="right")
plt.ylabel("Score moyen")
plt.ylim(0, 1.0)
plt.title("Comparaison IoU / Dice par vêtement — Présentes vs Pondérées", fontsize=13, weight="bold")
plt.legend(ncol=2)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# --- Tableau récapitulatif (arrondi conseillé) ---
# print(df_moy.round(3).to_string(index=False))


## Performance par image — classes présentes

Ce graphique montre la qualité de segmentation **par image**, en ne considérant que les classes présentes dans chaque image.

Métriques affichées :

- **mIoU** (présentes)
- **mDice** (présentes)

**Objectif du graphique :**
- Évaluer la stabilité du modèle image par image
- Détecter d'éventuelles images difficiles (contre-jours, poses complexes, accessoires…)

**Lecture :**
- Barres plus hautes ➜ images bien segmentées
- Variations ➜ dépendance au contexte (lumière, taille du sujet, angle de vue)

<p><i> Ce graphique aide à repérer les cas complexes qui méritent une analyse qualitative.</i></p>


In [ ]:
df_img_rows = []
for img, g in df.groupby("Nom_Image", sort=False):
    g_pres = g[g["Support"] > 0]
    if g_pres.empty:
        continue

    iou  = g_pres["IoU"].to_numpy(dtype=float)
    dice = g_pres["Dice"].to_numpy(dtype=float)
    sup  = g_pres["Support"].to_numpy(dtype=float)

    mIoU_pres = moyenne_presente(iou, sup)
    mDice_pres = moyenne_presente(dice, sup)
    mIoU_pond  = moyenne_ponderee(iou, sup)
    mDice_pond = moyenne_ponderee(dice, sup)

    df_img_rows.append({
        "Nom_Image": img,
        "mIoU_Presente": mIoU_pres,
        "mDice_Presente": mDice_pres,
        "mIoU_Ponderee": mIoU_pond,
        "mDice_Ponderee": mDice_pond,
        "classes_GT": int((sup > 0).sum())
    })

df_img = pd.DataFrame(df_img_rows)

# --- Bar chart (Présentes) ---
x = np.arange(len(df_img))
bar = 0.4

plt.figure(figsize=(12, 5))
plt.bar(x - bar/2, df_img["mIoU_Presente"],  width=bar, label="mIoU",  color="#4e79a7")
plt.bar(x + bar/2, df_img["mDice_Presente"], width=bar, label="mDice", color="#f28e2b")

plt.xticks(x, df_img["Nom_Image"], rotation=45, ha="right")
plt.ylabel("Score moyen (Présentes)")
plt.ylim(0, 1.0)
plt.title("Performance du modèle par image — Classes présentes", fontsize=13, weight="bold")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# --- Bar chart (Pondérées) ---
plt.figure(figsize=(12, 5))
plt.bar(x - bar/2, df_img["mIoU_Ponderee"],  width=bar, label="mIoU",  color="#4e79a7")
plt.bar(x + bar/2, df_img["mDice_Ponderee"], width=bar, label="mDice", color="#f28e2b")

plt.xticks(x, df_img["Nom_Image"], rotation=45, ha="right")
plt.ylabel("Score moyen (Pondérées)")
plt.ylim(0, 1.0)
plt.title("Performance du modèle par image — Classes pondérées", fontsize=13, weight="bold")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Conclusion générale

<strong>Verdict :</strong>
Le modèle <strong style="color:#2980B9;">SegFormer_B3_Clothes</strong> délivre une segmentation <em>fiable</em> et <em>cohérente</em> sur les vêtements principaux (pantalons, hauts, robes, jupes).
Les scores moyens (mIoU / mDice) confirment une <strong style="color:#27AE60;">robustesse globale</strong> sur des scènes variées, avec une sensibilité attendue aux <em>petits accessoires</em>.

<ul> <li><strong>Forces :</strong> <span style="color:#27AE60;">trés bonnes performances</span> sur les classes dominantes.</li> <li><strong>Limites :</strong> <span style="color:#E67E22;">baisse sur accessoires</span> (petite surface, contours fins).</li> <li><strong>Utilisation :</strong> adaptée au <strong>e-commerce</strong>, <strong>catalogage</strong>, <strong>essayage virtuel</strong>, <strong>analytics mode</strong>.</li> </ul>

### Conclusion détaillée
<span style="color:#27AE60;">Points forts</span>
<ul> <li>Très bonne précision sur <strong>pantalon</strong>, <strong>haut</strong>, <strong>robe</strong>, <strong>jupe</strong></li> <li>Sensible aux variations de pose, lumière et arrière-plan</li> <li>Scores pondérés élevés → excellente performance sur grandes surfaces de vêtements</li> </ul>

### Limites observées
<ul> <li>Accessoires difficiles : <em>chapeau, ceinture, chaussures, écharpe</em></li> <li>Scènes complexes : ombres dures, textures proches</li> <li><strong>Bords d’image</strong> : zones latérales et pieds parfois mal segmentés</li> <li><strong>Distinction doite et gauche </strong> : Le modèle présente des difficultés récurrentes à différencier les parties du corps situées à droite et à gauche, en particulier sur les zones latérales.</li></ul>

### Recommandations
<ol> <li><strong>Enrichir les données</strong>&nbsp;: plus d'accessoires et de scènes complexes</li> <li><strong>Rééquilibrer les classes</strong> peu représentées</li> <li><strong>Affiner les contours</strong> via post-traitement</li> <li><strong>Renforcer le training sur bords d’image</strong> (éléments coupés)</li> <li><strong>Approche hybride</strong> : modèle principal + module accessoires</li> </ol>

### Lecture des graphiques
<table> <thead> <tr><th>Graphique</th><th>Message</th><th>Rôle</th></tr> </thead> <tbody> <tr><td>mIoU / mDice global</td><td><span style="color:#27AE60;">Performance robuste</span></td><td>Validation générale</td></tr> <tr><td>Scores par vêtement</td><td><span style="color:#2980B9;">Différences selon classes</span></td><td>Priorités d’amélioration</td></tr> <tr><td>Scores par image</td><td><span style="color:#E67E22;">Variabilité selon contexte</span></td><td>Cas difficiles</td></tr> <tr><td>Pondération par surface</td><td><span style="color:#27AE60;">Grands objets maîtrisés</span></td><td>Impact de la taille</td></tr> </tbody> </table>

## Conclusion finale
<p> Le modèle <strong style="color:#2980B9;">SegFormer_B3_Clothes</strong> constitue une <strong style="color:#27AE60;">base solide</strong> pour la segmentation vestimentaire. Il excelle sur les vêtements principaux, et les ajustements proposés permettront d'améliorer la gestion des petits accessoires et des zones en bord d’image. </p>


### Ouverture : mise en valeur & perspectives
<p> Dans une logique d’évolution continue, cette solution d’IA peut être adaptée à votre univers de marque : </p> <ul> <li><strong style="color:#2980B9;">Apprentissage sur vos collections</strong> (styles, matières, coloris)</li> <li><strong>Alignement avec votre identité visuelle</strong></li> <li><strong>Détection avancée</strong> de détails métiers (textures, coupes)</li> <li><strong>Extension aux accessoires</strong> : chaussures, sacs, bijoux, lunettes</li> <li><strong>Amélioration continue</strong> via retours utilisateurs et catalogue interne</li> </ul> <p> Cela fait évoluer le modèle en un <strong style="color:#27AE60;">outil sur-mesure</strong> permettant d'accélérer la mise en ligne, fiabiliser le tagging produit et automatiser le traitement visuel. </p>

<strong>Analyse du coût — Traitement de 500 000 images avec l'IA</strong>
<strong>Objectif :</strong>

### Évaluer le coût pour analyser 
<strong>500 000 images</strong> avec un modèle d’IA de segmentation vestimentaire, en s’appuyant sur l’infrastructure Cloud de <strong style="color:#E67E22;">Hugging Face</strong>.</p> <p>Ce traitement répond aux besoins des entreprises souhaitant automatiser :</p> <ul>
    <li>Le <strong>catalogage produit</strong></li>
    <li>L’<strong>analyse photo e-commerce provenant d'internet</strong></li>
    <li>La préparation de <strong>datasets IA internes</strong></li>
    <li>Le nettoyage et la <strong>classification d’archives visuelles</strong></li> </ul>

### <strong>Résumé décisionnel</strong>
<table> <tr> <td><strong>Budget estimé</strong></td> <td><strong style="color:#2ECC71;">30 € – 70 €</strong></td> </tr> <tr> <td><strong>Durée du traitement</strong></td> <td>≈ <strong>1 à 2 jours</strong></td> </tr> <tr> <td><strong>Volume</strong></td> <td><strong>500 000 images</strong></td> </tr> <tr> <td><strong>Solution</strong></td> <td>GPU dédié (<a href="https://huggingface.co/inference-endpoints" target="_blank">Hugging Face Endpoints</a>)</td> </tr> <tr> <td><strong>Abonnement inclus</strong></td> <td><a href="https://huggingface.co/pricing" target="_blank"><strong>9 $/mois (~8,50 €)</strong></a></td> </tr> </table>

### <span><strong>Avantages</strong></span>

<ul style="list-style: none; padding-left: 0;">
    <li><strong>Très économique</strong> pour un volume industriel</li>
    <li><strong>Scalable</strong> (traitement millions d’images possible)</li>
    <li><strong>Sans infrastructure à gérer</strong></li> </ul>

### <strong>Détail des coûts</strong>

<span><strong>Option A — Paiement à l’usage</span></strong> <br>
<i>(idéal pour tests / usage ponctuel)</i><br><br>

<a href="https://huggingface.co/docs/inference-providers/pricing" target="_blank">Tarifs API Hugging Face</a>

<table> <thead> <tr> <th>Scénario</th> <th style="color:#2ECC71;">Coût inférence</th> <th>Abonnement</th> <th style="color:#2ECC71;">Total</th> <th>Commentaire</th> </tr> </thead> <tbody> <tr> <td>Rapide</td> <td>~27 $</td> <td>9 $</td> <td><strong style="color:#2ECC71;">~36 $</strong></td> <td>Flux optimal</td> </tr> <tr> <td>Standard</td> <td>~54 $</td> <td>9 $</td> <td><strong style="color:#2ECC71;">~63 $</strong></td> <td>Cas réaliste</td> </tr> <tr> <td>Prudent</td> <td>~90 $</td> <td>9 $</td> <td><strong style="color:#2ECC71;">~99 $</strong></td> <td>Images lourdes</td> </tr> </tbody> </table>

### <strong>Calculs simples pour le client</strong>
<ul><li>Prix par image (sur base 63 $)</li>
<li>500 000 images</li>
<li>Coût par image = 63 / 500 000 = 0.000126 $≈ 0,00012 $ / image</li>
<li>Soit 1 000 000 images = 1 000 000 × 0,00012 $ = 126 $</li>
</ul>
<span><i>Prix d’une image ≃ 0,00011–0,00020 $ selon scénario (≈ 0,01 centime — coût insignifiant à grande échelle)</i></span>

### <strong>Besoin en stockage</strong>
<table> <tr><td>Images</td><td>~250 Go</td></tr> <tr><td>Masques IA</td><td>~150–300 Go</td></tr><tr><td>Logs</td><td>~10 Go</td></tr> <tr> <td><strong>Total</strong></td> <td><strong style="color:#2ECC71;">~400–600 Go</strong></td> </tr> </table>

### <strong>Conclusion & Recommandation</strong>
<table> <tr> <td><strong>Coût total</strong></td> <td><strong style="color:#2ECC71;">30–70 €</strong></td> </tr> <tr> <td><strong>Durée</strong></td> <td><strong>1–2 jours</strong></td> </tr> <tr> <td><strong>Matériel</strong></td> <td>🚫 Aucun — Cloud managé</td> </tr> </table> 
<p><strong>Recommandation :</strong> utiliser un <strong style="color:#27AE60;">serveur Hugging Face dédié</strong> (~30–60 €) pour un traitement fluide et fiable.</p> <p><i>Cette solution offre un traitement industriel, rapide, flexible et très économique, sans investissement matériel.</i></p>